# In this notebook, we will fit columns to pipelines and will train different ml models. Best one will be dumped as main model.

In [39]:
import numpy as np
import pandas as pd

In [40]:
data = pd.read_csv(r'..\Datasets\Cleaned\Data_cleaned.csv')

In [41]:
data.columns

Index(['Restaurant_latitude', 'Restaurant_longitude',
       'Delivery_location_latitude', 'Delivery_location_longitude',
       'Weatherconditions', 'Road_traffic_density', 'Vehicle_condition',
       'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival',
       'City', 'Time_taken(min)', 'distance_km'],
      dtype='str')

In [42]:
numeric_cols = data.select_dtypes(include='number')
numeric_cols

,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Vehicle_condition,multiple_deliveries,Time_taken(min),distance_km
0,22.745049,75.892471,22.765049,75.912471,2,0,24,3.03
1,12.913041,77.683237,13.043041,77.813237,2,1,33,20.18
2,12.914264,77.678400,12.924264,77.688400,0,1,26,1.55
3,11.003669,76.976494,11.053669,77.026494,0,1,21,7.79
4,12.972793,80.249982,13.012793,80.289982,1,1,30,6.21
...,...,...,...,...,...,...,...,...
45588,26.902328,75.794257,26.912328,75.804257,1,0,32,1.49
45589,18.551440,75.898497,18.640963,76.011991,0,1,36,15.56
45590,13.022394,80.242439,13.052394,80.272439,1,0,16,4.66
45591,11.001753,76.986241,11.041753,77.026241,0,1,26,6.23


In [43]:
str_col = data.select_dtypes(include='str')
str_col

,Weatherconditions,Road_traffic_density,Type_of_order,Type_of_vehicle,Festival,City
0,Sunny,High,Snack,motorcycle,No,Urban
1,Stormy,Jam,Snack,scooter,No,Metropolitian
2,Sandstorms,Low,Drinks,motorcycle,No,Urban
3,Sunny,Medium,Buffet,motorcycle,No,Metropolitian
4,Cloudy,High,Snack,scooter,No,Metropolitian
...,...,...,...,...,...,...
45588,Windy,High,Meal,motorcycle,No,Metropolitian
45589,Windy,Jam,Buffet,motorcycle,No,Metropolitian
45590,Cloudy,Low,Drinks,scooter,No,Metropolitian
45591,Cloudy,High,Snack,motorcycle,No,Metropolitian


In [45]:
from sklearn.model_selection import train_test_split

skewed_col = ['distance_km']
numeric_cols = [
    'Restaurant_latitude', 'Restaurant_longitude',
    'Delivery_location_latitude', 'Delivery_location_longitude',
    'Vehicle_condition', 'multiple_deliveries'
]
alpha_col = [
    'Weatherconditions', 'Type_of_order', 'Type_of_vehicle', 
    'Festival', 'City'
]
ordinal_col = ['Road_traffic_density']

# Use alpha_col here instead of str_col
feature_col = skewed_col + numeric_cols + ordinal_col + alpha_col

X = data[feature_col]
y = data['Time_taken(min)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

In [46]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
#1. Skew
skew_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())

])

#2. Numeric Features
plain_numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])

#2. Ordinal
ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Jam']]))
])

#3. Nominal Features
alpha_pipeline = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown="ignore"))
])


    # (konsi_pipeline, konsa_feature)
preprocessor = ColumnTransformer(transformers=[
    ("Skewed_Pipeline", skew_pipeline, skewed_col),
    ("Plain_Numeric",plain_numeric_pipeline, numeric_cols ),
    ('Ordinal', ordinal_pipeline, ordinal_col),
    ('Normal', alpha_pipeline, alpha_col)
])
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Skewed_Pipeline', ...), ('Plain_Numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_

In [47]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
lr_preds       = lr_pipeline.predict(X_test)
lr_preds_train = lr_pipeline.predict(X_train)

lr_r2_testing  = r2_score(y_test, lr_preds)
lr_r2_training = r2_score(y_train, lr_preds_train)
lr_mae         = mean_absolute_error(y_test, lr_preds)

print(f"Accuracy of Training {lr_r2_training}")
print(f"Accuracy of Testing {lr_r2_testing}")
print(f'MAE {lr_mae}')

Accuracy of Training 0.4534045993849193
Accuracy of Testing 0.45725891210554803
MAE 5.579979297734275


In [49]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random forest', RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_preds          = rf_pipeline.predict(X_test)
rf_preds_training = rf_pipeline.predict(X_train)

rf_r2_testing  = r2_score(y_test, rf_preds)
rf_r2_training = r2_score(y_train, rf_preds_training)
rf_mae         = mean_absolute_error(y_test, rf_preds)

print(f"Accuracy of Training {rf_r2_training}")
print(f"Accuracy of Testing {rf_r2_testing}")
print(f"MAE : {rf_mae}")

Accuracy of Training 0.9280752643844739
Accuracy of Testing 0.5825038580991979
MAE : 4.713998089448427


In [50]:
from sklearn.model_selection import RandomizedSearchCV
param_grid = {
    'random forest__n_estimators' : [100,200,300],
    'random forest__max_depth'   : [5,10,15],
    'random forest__min_samples_split':[2,5,10],
    'random forest__min_samples_leaf': [1,2,4]
}
random_search=RandomizedSearchCV(estimator=rf_pipeline,
                                 param_distributions=param_grid,
                                 n_iter=15,
                                 cv=5,
                                 scoring='r2',
                                 random_state=42,
                                 n_jobs=-1)
random_search.fit(X_train,y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'random forest__max_depth': [5, 10, ...], 'random forest__min_samples_leaf': [1, 2, ...], 'random forest__min_samples_split': [2, 5, ...], 'random forest__n_estimators': [100, 200, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` dire

In [51]:
random_search.best_params_

{'random forest__n_estimators': 300,
 'random forest__min_samples_split': 10,
 'random forest__min_samples_leaf': 1,
 'random forest__max_depth': 10}

In [58]:
rf_best_pipeline = random_search.best_estimator_

# Predict on both sets
rf_best_preds_train = rf_best_pipeline.predict(X_train)
rf_best_preds_test  = rf_best_pipeline.predict(X_test)

# Evaluate matching sets
print(f"R2 Score (Train) : {r2_score(y_train, rf_best_preds_train):.4f}")
print(f"R2 Score (Test)  : {r2_score(y_test, rf_best_preds_test):.4f}")
print(f"MAE Score (Test) : {mean_absolute_error(y_test, rf_best_preds_test):.4f}")

R2 Score (Train) : 0.6658
R2 Score (Test)  : 0.6313
MAE Score (Test) : 4.4813


In [54]:
from xgboost import XGBRegressor

# Define the XGBoost Pipeline
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('xgboost', XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        n_jobs=-1
    ))
])

# Fit on training data
xgb_pipeline.fit(X_train, y_train)

# Predictions
xgb_preds_training = xgb_pipeline.predict(X_train)
xgb_preds          = xgb_pipeline.predict(X_test)

# Evaluation metrics
xgb_r2_training = r2_score(y_train, xgb_preds_training)
xgb_r2_testing  = r2_score(y_test, xgb_preds)
xgb_mae         = mean_absolute_error(y_test, xgb_preds)

# Output results
print(f"Accuracy of Training (R2) : {xgb_r2_training:.4f}")
print(f"Accuracy of Testing (R2)  : {xgb_r2_testing:.4f}")
print(f"MAE                       : {xgb_mae:.4f}")

Accuracy of Training (R2) : 0.6652
Accuracy of Testing (R2)  : 0.6325
MAE                       : 4.4776


In [55]:
param_distributions = {
    'xgboost__n_estimators': [100, 200, 300, 500],
    'xgboost__max_depth': [3, 4, 5, 6, 8, 10],
    'xgboost__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'xgboost__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'xgboost__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'xgboost__min_child_weight': [1, 3, 5, 7],
    'xgboost__gamma': [0, 0.1, 0.2, 0.4]
}

# 3. Setup RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=25,               # Total hyperparameter combinations to try
    scoring='neg_mean_absolute_error',  # Optimization target
    cv=5,                    # 5-fold cross-validation
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# 4. Execute search on training set
random_search.fit(X_train, y_train)

# 5. Extract best model and evaluate
best_xgb_model = random_search.best_estimator_

xgb_tuned_preds_train = best_xgb_model.predict(X_train)
xgb_tuned_preds_test  = best_xgb_model.predict(X_test)

tuned_r2_train = r2_score(y_train, xgb_tuned_preds_train)
tuned_r2_test  = r2_score(y_test, xgb_tuned_preds_test)
tuned_mae      = mean_absolute_error(y_test, xgb_tuned_preds_test)

print("\n--- Best Parameters ---")
for param, val in random_search.best_params_.items():
    print(f"{param}: {val}")

print("\n--- Tuned XGBoost Performance ---")
print(f"Accuracy of Training (R2) : {tuned_r2_train:.4f}")
print(f"Accuracy of Testing (R2)  : {tuned_r2_test:.4f}")
print(f"MAE                       : {tuned_mae:.4f}")

Fitting 5 folds for each of 25 candidates, totalling 125 fits

--- Best Parameters ---
xgboost__subsample: 1.0
xgboost__n_estimators: 100
xgboost__min_child_weight: 1
xgboost__max_depth: 6
xgboost__learning_rate: 0.1
xgboost__gamma: 0
xgboost__colsample_bytree: 0.7

--- Tuned XGBoost Performance ---
Accuracy of Training (R2) : 0.6658
Accuracy of Testing (R2)  : 0.6313
MAE                       : 4.4813


In [60]:
import joblib

# Replace with your fitted XGBoost pipeline variable name
best_pipeline = xgb_pipeline  

joblib.dump(best_pipeline, r'..\delivery_time_xgb_pipeline.joblib')
print("Model pipeline exported successfully.")

Model pipeline exported successfully.
